# List 3
## Author: Jan Pułtorak

In [10]:
import torch
import numpy as np
import random
import sklearn
from tqdm import tqdm
import matplotlib.pyplot as plt
import torchvision
from torchvision.transforms import v2

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed);

In [89]:
train_data = torchvision.datasets.FashionMNIST(root='../data', train=True, download=True)
test_data = torchvision.datasets.FashionMNIST(root='../data', train=False, download=True)

train_data.data.shape, test_data.data.shape, train_data.class_to_idx

(torch.Size([60000, 28, 28]),
 torch.Size([10000, 28, 28]),
 {'T-shirt/top': 0,
  'Trouser': 1,
  'Pullover': 2,
  'Dress': 3,
  'Coat': 4,
  'Sandal': 5,
  'Shirt': 6,
  'Sneaker': 7,
  'Bag': 8,
  'Ankle boot': 9})

In [85]:
def shuffle(x: torch.Tensor):
    return x[torch.randperm(len(x))]

def prepare_binary_data(dataset: torchvision.datasets.FashionMNIST, positive_class=5):
    X_raw  = dataset.data.float()
    X_raw = X_raw.reshape((X_raw.shape[0], -1))

    # print(X_raw.shape)
    y = dataset.targets

    pos_idx = torch.nonzero(y == positive_class).flatten()
    neg_idx = torch.nonzero(y != positive_class).flatten()


    total_pos = len(pos_idx)
    neg_idx_sampled = shuffle(neg_idx)[:total_pos]

    # print(pos_idx.shape, neg_idx_sampled.shape)
    # print(pos_idx[:10], neg_idx_sampled[:10])

    all_idx = shuffle(torch.cat([pos_idx, neg_idx_sampled]))
    # print(all_idx.shape)

    X = X_raw[all_idx]
    X_norm = X / 255.0

    y = (y[all_idx] == positive_class).float().reshape(-1, 1)
    # print(y.shape)

    return X, X_norm, y



In [86]:
X_train_raw, X_train_norm, y_train = prepare_binary_data(train_data)
X_test_raw, X_tes_norm, y_val = prepare_binary_data(test_data)